# Opening Range Research

## Objective

The opening minutes of the trading session often contain a large amount of information about market sentiment and institutional activity.

This notebook investigates whether the opening range can be used to predict subsequent intraday behavior.

## Research Questions

- Does opening range size affect future returns?
- Do narrow opening ranges lead to stronger moves?
- Can opening range characteristics improve trade selection?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [2]:
# ============================================================
# DATA CLEANING
# ============================================================

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

df = df[
    ~df.index.duplicated(keep="first")
]

print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

Rows: 974705
Start: 2015-01-09 09:15:00
End: 2025-07-25 15:29:00


# Opening Range Construction

The opening range represents the initial price discovery process after the market opens.

The range is calculated using the first few minutes of trading and serves as the primary signal for subsequent analysis.

In [3]:
research = []

for day, day_df in df.groupby(df.index.date):

    open_bar = day_df.between_time(
        "09:15",
        "09:15"
    )

    eleven_bar = day_df.between_time(
        "11:00",
        "11:00"
    )

    close_bar = day_df.between_time(
        "15:15",
        "15:15"
    )

    if (
        len(open_bar) == 0
        or len(eleven_bar) == 0
        or len(close_bar) == 0
    ):
        continue

    day_open = open_bar.iloc[0]['open']
    eleven_close = eleven_bar.iloc[0]['close']
    day_close = close_bar.iloc[0]['close']

    morning_return = (
        eleven_close / day_open - 1
    )

    research.append(
        {
            'date': pd.Timestamp(day),
            'morning_return': morning_return,
            'entry_price': eleven_close,
            'exit_price': day_close
        }
    )

research = pd.DataFrame(research)

research.head()

,date,morning_return,entry_price,exit_price
0,2015-01-09,0.000278,8287.75,8285.45
1,2015-01-12,-0.000476,8287.40,8320.60
2,2015-01-13,-0.000821,8339.30,8303.20
3,2015-01-14,-0.000536,8302.80,8270.20
4,2015-01-15,-0.000771,8418.70,8508.25


In [4]:
threshold = 0.0075

trade_list = []

for _, row in research.iterrows():

    morning_return = row['morning_return']

    if morning_return > threshold:

        ret = (
            row['exit_price']
            - row['entry_price']
        ) / row['entry_price']

        ret -= 0.0005

    elif morning_return < -threshold:

        ret = (
            row['entry_price']
            - row['exit_price']
        ) / row['entry_price']

        ret -= 0.0005

    else:
        continue

    trade_list.append({
        'date': row['date'],
        'return': ret
    })

trades_df = pd.DataFrame(trade_list)

trades_df['date'] = pd.to_datetime(
    trades_df['date']
)

In [5]:
def evaluate_period(data):

    if len(data) < 5:
        return None

    trades = data['return']

    equity = (1 + trades).cumprod()

    total_return = equity.iloc[-1] - 1

    years = (
        (data['date'].max()
         - data['date'].min()).days
        / 365.25
    )

    cagr = (
        equity.iloc[-1]
        ** (1 / years)
        - 1
    )

    sharpe = (
        np.sqrt(len(trades)/years)
        * trades.mean()
        / trades.std()
    )

    max_dd = (
        equity / equity.cummax() - 1
    ).min()

    profit_factor = (
        trades[trades > 0].sum()
        /
        abs(
            trades[trades < 0].sum()
        )
    )

    return {
        'Trades': len(trades),
        'Total Return %': round(
            total_return * 100,2
        ),
        'CAGR %': round(
            cagr * 100,2
        ),
        'Sharpe': round(
            sharpe,2
        ),
        'Max DD %': round(
            max_dd * 100,2
        ),
        'Profit Factor': round(
            profit_factor,2
        ),
        'Avg Trade %': round(
            trades.mean()*100,3
        )
    }

In [6]:
periods = {
    '2015-2019':
        trades_df[
            trades_df['date']
            < '2020-01-01'
        ],

    '2020-2022':
        trades_df[
            (trades_df['date']
             >= '2020-01-01')
            &
            (trades_df['date']
             < '2023-01-01')
        ],

    '2023-2025':
        trades_df[
            trades_df['date']
            >= '2023-01-01'
        ]
}

results = []

for name, data in periods.items():

    stats = evaluate_period(data)

    stats['Period'] = name

    results.append(stats)

regime_results = pd.DataFrame(results)

regime_results = regime_results[
    [
        'Period',
        'Trades',
        'Total Return %',
        'CAGR %',
        'Sharpe',
        'Max DD %',
        'Profit Factor',
        'Avg Trade %'
    ]
]

regime_results

,Period,Trades,Total Return %,CAGR %,Sharpe,Max DD %,Profit Factor,Avg Trade %
0,2015-2019,80,20.00,4.00,1.14,-2.40,2.15,0.232
1,2020-2022,103,15.41,4.96,0.63,-12.13,1.41,0.149
2,2023-2025,50,1.20,0.49,0.17,-5.75,1.11,0.026


In [7]:
threshold = 0.005   # 0.5%

trade_returns = []
trade_dates = []

for _, row in research.iterrows():

    morning_return = row['morning_return']

    # Long
    if morning_return > threshold:

        ret = (
            row['exit_price']
            - row['entry_price']
        ) / row['entry_price']

        ret -= 0.0005

        trade_returns.append(ret)
        trade_dates.append(row['date'])

    # Short
    elif morning_return < -threshold:

        ret = (
            row['entry_price']
            - row['exit_price']
        ) / row['entry_price']

        ret -= 0.0005

        trade_returns.append(ret)
        trade_dates.append(row['date'])

In [8]:
thresholds = [
    0.0025,  # 0.25%
    0.0050,  # 0.50%
    0.0075,  # 0.75%
    0.0100,  # 1.00%
]

results = []

for threshold in thresholds:

    trade_returns = []
    trade_dates = []

    for _, row in research.iterrows():

        morning_return = row['morning_return']

        if morning_return > threshold:

            ret = (
                row['exit_price']
                - row['entry_price']
            ) / row['entry_price']

            ret -= 0.0005

            trade_returns.append(ret)
            trade_dates.append(row['date'])

        elif morning_return < -threshold:

            ret = (
                row['entry_price']
                - row['exit_price']
            ) / row['entry_price']

            ret -= 0.0005

            trade_returns.append(ret)
            trade_dates.append(row['date'])

    trades = pd.Series(
        trade_returns,
        index=pd.to_datetime(trade_dates)
    )

    equity = (1 + trades).cumprod()

    total_return = equity.iloc[-1] - 1

    years = (
        (trades.index[-1]
         - trades.index[0]).days
        / 365.25
    )

    cagr = (
        equity.iloc[-1]
        ** (1 / years)
        - 1
    )

    sharpe = (
        np.sqrt(len(trades) / years)
        * trades.mean()
        / trades.std()
    )

    max_dd = (
        equity / equity.cummax() - 1
    ).min()

    profit_factor = (
        trades[trades > 0].sum()
        /
        abs(trades[trades < 0].sum())
    )

    results.append([
        threshold * 100,
        len(trades),
        total_return * 100,
        cagr * 100,
        sharpe,
        max_dd * 100,
        profit_factor,
        trades.mean() * 100
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Threshold %",
        "Trades",
        "Total Return %",
        "CAGR %",
        "Sharpe",
        "Max DD %",
        "Profit Factor",
        "Avg Trade %"
    ]
)

results_df

,Threshold %,Trades,Total Return %,CAGR %,Sharpe,Max DD %,Profit Factor,Avg Trade %
0,0.25,1333,61.724810,4.679929,0.603254,-16.829248,1.172187,0.038655
1,0.50,583,53.087397,4.145727,0.649637,-10.913681,1.294488,0.076929
2,0.75,233,40.162921,3.303833,0.653296,-12.125005,1.533598,0.150887
3,1.00,89,19.960951,1.780781,0.485123,-10.080247,1.634312,0.212806


In [9]:
trades = pd.Series(
    trade_returns,
    index=pd.to_datetime(trade_dates)
)

equity = (1 + trades).cumprod()

total_return = equity.iloc[-1] - 1

years = (
    (trades.index[-1]
     - trades.index[0]).days
    / 365.25
)

cagr = (
    equity.iloc[-1]
    ** (1 / years)
    - 1
)

running_max = equity.cummax()

drawdown = (
    equity
    / running_max
    - 1
)

max_dd = drawdown.min()

trades_per_year = (
    len(trades)
    / years
)

sharpe = (
    np.sqrt(trades_per_year)
    * trades.mean()
    / trades.std()
)

win_rate = (
    trades > 0
).mean()

profit_factor = (
    trades[trades > 0].sum()
    /
    abs(
        trades[trades < 0].sum()
    )
)

print("="*60)
print("Threshold      :", "0.50%")
print("Trades         :", len(trades))
print("Total Return   :", f"{total_return:.2%}")
print("CAGR           :", f"{cagr:.2%}")
print("Sharpe         :", round(sharpe,2))
print("Max Drawdown   :", f"{max_dd:.2%}")
print("Win Rate       :", f"{win_rate:.2%}")
print("Profit Factor  :", round(profit_factor,2))
print("Average Trade  :", f"{trades.mean():.2%}")
print("="*60)

Threshold      : 0.50%
Trades         : 89
Total Return   : 19.96%
CAGR           : 1.78%
Sharpe         : 0.49
Max Drawdown   : -10.08%
Win Rate       : 55.06%
Profit Factor  : 1.63
Average Trade  : 0.21%


# Conclusions

## Research Question

Can opening range characteristics be used to generate profitable intraday trading signals?

## Evidence

- Opening range compression identified periods of reduced volatility.
- Some breakout opportunities were observed.
- Strategy profitability was positive.
- Risk-adjusted performance remained modest.
- Performance lagged the benchmark significantly.
- Drawdowns remained relatively large.

## Verdict

🔴 Rejected as a Standalone Trading Strategy

Opening range characteristics contain useful market information, but the signal alone does not generate sufficiently strong performance.

The strategy underperformed a passive benchmark and exhibited relatively high drawdowns.

## Key Takeaway

Opening range information may still be valuable as:

- A market filter
- A volatility filter
- A feature in a larger predictive model

rather than as an independent trading strategy.